# Steering Vector Application

This 50-minute-track notebook is intentionally self-contained: it uses a seeded tiny PyTorch model, no downloads, and only public TorchLens APIs. The goal is to show the full workflow shape you would use on a larger model while keeping every cell runnable on CPU.


Steering adds a direction to a saved activation and propagates the effect. In larger language models, the direction may come from contrastive prompts or a probe. Here we compute a local clean-minus-reference direction so the recipe is deterministic.


In [ ]:
from __future__ import annotations

import math
from typing import Any

import matplotlib.pyplot as plt
import torch
from torch import nn

import torchlens as tl

SEED = 1607
torch.manual_seed(SEED)
torch.set_grad_enabled(False)


class TinyBlock(nn.Module):
    """One residual MLP block used as a transformer-sized stand-in."""

    def __init__(self, width: int) -> None:
        """Initialize the block."""
        super().__init__()
        self.ln = nn.LayerNorm(width)
        self.up = nn.Linear(width, width)
        self.down = nn.Linear(width, width)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Apply layer norm, MLP, and residual addition."""
        hidden = torch.relu(self.up(self.ln(x)))
        return x + self.down(hidden)


class TinyTransformer(nn.Module):
    """Small token model with named blocks and sequence positions."""

    def __init__(self, vocab_size: int = 16, width: int = 12, depth: int = 3) -> None:
        """Initialize embeddings, blocks, and unembedding head."""
        super().__init__()
        self.embed = nn.Embedding(vocab_size, width)
        self.blocks = nn.ModuleList([TinyBlock(width) for _ in range(depth)])
        self.unembed = nn.Linear(width, vocab_size, bias=False)

    def forward(self, tokens: torch.Tensor) -> torch.Tensor:
        """Return logits for every token position."""
        x = self.embed(tokens)
        for block in self.blocks:
            x = block(x)
        return self.unembed(x)

In [ ]:
model = TinyTransformer(vocab_size=14, width=10, depth=2).eval()
positive_tokens = torch.tensor([[1, 2, 3, 4]])
negative_tokens = torch.tensor([[4, 3, 2, 1]])
base_tokens = torch.tensor([[1, 2, 2, 4]])
positive = tl.log_forward_pass(
    model, positive_tokens, vis_opt="none", intervention_ready=True, name="positive"
)
negative = tl.log_forward_pass(
    model, negative_tokens, vis_opt="none", intervention_ready=True, name="negative"
)
base = tl.log_forward_pass(model, base_tokens, vis_opt="none", intervention_ready=True, name="base")
site = base.find_sites(tl.func("relu")).first()
direction = positive[site.layer_label].activation - negative[site.layer_label].activation
print(site.layer_label, tuple(direction.shape))

In [ ]:
steered = base.fork("steered")
steered.attach_hooks(tl.label(site.layer_label), tl.steer(direction, magnitude=0.25)).replay()

base_logits = base.layer_list[-1].activation
steered_logits = steered.layer_list[-1].activation
print(f"logit delta norm: {torch.linalg.vector_norm(steered_logits - base_logits):.4f}")
print(f"base final-token argmax: {int(base_logits[0, -1].argmax())}")
print(f"steered final-token argmax: {int(steered_logits[0, -1].argmax())}")

Use small magnitudes first. A direction with the exact activation shape can steer specific positions; a broadcastable direction can steer an entire site. The notebook uses the exact shape to make the intervention unambiguous.
